<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 03. Creación de Ratios, Interacciones y Transformaciones
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 06
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/06%20-%20Feature%20Engineering/Para%20Dummies/03_Creacion_de_Caracteristicas_Feature_Engineering_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué significa "crear" una característica nueva? 🏗️

Cuando hablamos del clima, casi nunca decimos "hoy hay 30°C y 80% de humedad". Decimos algo más simple: "hoy se siente pesado" o "la sensación térmica es de 34°C". Esa "sensación térmica" no la mide ningún termómetro directamente: alguien la **calculó combinando** la temperatura, la humedad y el viento en una sola cifra que es mucho más útil para decidir si llevar sombrilla o no.

Eso es exactamente **crear características (*feature engineering*)**: combinar los datos que ya tenemos, con una fórmula sencilla, para fabricar una columna nueva que le cuente al modelo algo que las columnas originales no decían tan claro.

En este cuaderno veremos, con ejemplos pequeños, las cuatro formas más comunes de crear características:
1. **Ratios y proporciones** (dividir una columna entre otra).
2. **Transformaciones matemáticas** como el logaritmo (para "domar" datos muy desiguales).
3. **Conteos** (sumar cuántas cosas de un tipo aparecen).
4. **Combinar/separar texto** y **resumir por grupo** (por ejemplo, "el promedio de mi ciudad").

---
## 1. Ratios: cuando dividir cuenta más que restar 📐

Imagina que quieres comparar dos apartamentos:
- Apartamento A: 300 millones de pesos, 60 m².
- Apartamento B: 500 millones de pesos, 150 m².

¿Cuál es "más caro"? Si solo miras el precio, B parece peor negocio. Pero si divides **precio entre metros cuadrados**, obtienes el precio *por metro*, que es la comparación justa: A cuesta 5 millones/m² y B cuesta ~3.3 millones/m². ¡B es más barato por metro, aunque cueste más en total!

Ese es el poder de un **ratio**: convierte dos números en uno solo que sí se puede comparar directamente entre filas.

In [ ]:
import pandas as pd
import numpy as np

# Un mini dataset de apartamentos
df_aptos = pd.DataFrame({
    'Apartamento': ['A', 'B', 'C', 'D'],
    'Precio_millones': [300, 500, 220, 640],
    'Area_m2': [60, 150, 45, 110]
})

# Creamos el ratio: precio por metro cuadrado
df_aptos['Precio_por_m2'] = df_aptos['Precio_millones'] / df_aptos['Area_m2']

df_aptos.sort_values('Precio_por_m2')

### 🤔 ¿Qué acaba de pasar?

- `Precio_millones / Area_m2` es una operación vectorizada: Pandas la aplica a las cuatro filas a la vez, sin necesidad de un bucle.
- Al ordenar por `Precio_por_m2`, vemos cuál apartamento es realmente la mejor oferta por metro — algo que **no** se ve mirando el precio total ni el área por separado.
- Esta misma idea es la que usan los bancos para calcular el ratio **deuda/ingreso** de una persona, o los ingenieros mecánicos para calcular la relación **peso/potencia** de un motor: dos columnas por separado dicen poco, pero su cociente dice mucho.

---
## 2. Logaritmo: aplanando montañas muy empinadas ⛰️

Los ingresos, los precios de las casas y muchas variables de dinero tienen un problema: la mayoría de los valores son "normales", pero unos pocos son gigantescos (piensa en los ingresos de casi todo el país frente a los de un puñado de millonarios). Esto se llama una distribución **sesgada** o **asimétrica**, y confunde a muchos modelos.

La solución clásica es aplicarle el **logaritmo** a la columna. Como algunos valores pueden ser cero, usamos `np.log1p(x)` (que calcula $\log(1+x)$ en lugar de $\log(x)$, para no romperse con el cero).

> 📌 **Piénsalo así:** el logaritmo "aplana" la montaña — comprime las distancias entre los números grandes y estira un poco las de los números pequeños, dejando una forma mucho más parecida a una campana.

In [ ]:
import matplotlib.pyplot as plt

# Simulamos ingresos muy sesgados (unos pocos valores enormes)
np.random.seed(0)
ingresos = np.random.lognormal(mean=14, sigma=1.0, size=1000)

log_ingresos = np.log1p(ingresos)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(ingresos / 1e6, bins=30, color='tomato', edgecolor='white')
axes[0].set_title('Ingresos originales (sesgados)')
axes[0].set_xlabel('Millones COP')

axes[1].hist(log_ingresos, bins=30, color='royalblue', edgecolor='white')
axes[1].set_title('log(1 + Ingresos)')
axes[1].set_xlabel('Escala logarítmica')
plt.tight_layout()
plt.show()

### 🤔 ¿Qué acaba de pasar?

- El histograma de la izquierda tiene una "cola" larga hacia la derecha: pocos ingresos gigantes estiran todo el gráfico.
- El histograma de la derecha, tras aplicar `np.log1p`, se parece mucho más a una campana simétrica.
- Muchos modelos (sobre todo los lineales) funcionan mejor cuando sus variables numéricas se parecen a esa campana, así que transformar con logaritmo suele mejorar el desempeño sin perder información: solo cambia la "escala" en la que medimos.

---
## 3. Conteos: contar banderas encendidas 🚩

A veces tienes muchas columnas de sí/no (por ejemplo: "¿había semáforo?", "¿había cruce peatonal?", "¿había reductor de velocidad?") y lo que realmente le importa al modelo no es cada bandera por separado, sino **cuántas** estaban encendidas en total.

En Python, `True` vale `1` y `False` vale `0`, así que sumar varias columnas booleanas te da directamente ese conteo.

In [ ]:
# Elementos de seguridad presentes en distintos cruces viales
df_cruces = pd.DataFrame({
    'Semaforo': [True, False, True, True],
    'Cruce_peatonal': [True, True, False, True],
    'Reductor_velocidad': [False, False, False, True],
})

# Sumamos a lo largo de las columnas (axis=1) para contar elementos presentes
df_cruces['Elementos_seguridad'] = df_cruces.sum(axis=1)

df_cruces

### 🤔 ¿Qué acaba de pasar?

- `df_cruces.sum(axis=1)` recorre **cada fila** y suma sus valores booleanos como si fueran 1 y 0.
- El resultado, `Elementos_seguridad`, es una sola columna que resume "qué tan protegido" está cada cruce, en lugar de que el modelo tenga que combinar tres columnas sueltas por su cuenta.
- Esta técnica es muy común con listas largas de síntomas, factores de riesgo o características de un producto: convertir "muchas banderas" en "un conteo" casi siempre ayuda.

---
## 4. Combinar y separar texto, y resumir por grupo 🔤📊

Dos trucos más, muy usados con datos de texto y categorías:

- **Separar una columna de texto en varias:** si tienes `"Corporativo L3"` en una sola columna, puedes partirla en `Tipo = "Corporativo"` y `Nivel = "L3"` usando `.str.split(" ")`.
- **Combinar columnas categóricas:** si sospechas que "ciudad" y "canal de venta" juntas dicen algo que por separado no dicen, puedes pegarlas en una sola columna con `+`.
- **Resumir por grupo (*group transform*):** por ejemplo, "el ingreso promedio de tu ciudad" — se calcula con `groupby(...).transform("mean")`, y le asigna a **cada fila** el promedio de su grupo, sin colapsar la tabla.

In [ ]:
df_clientes = pd.DataFrame({
    'Plan': ['Corporativo L3', 'Personal L1', 'Personal L3', 'Corporativo L2'],
    'Ciudad': ['Tunja', 'Bogota', 'Tunja', 'Cali'],
    'Ingreso': [4.2, 2.1, 3.0, 5.5]  # millones de pesos
})

# Separar texto en dos columnas nuevas
df_clientes[['Tipo_plan', 'Nivel_plan']] = df_clientes['Plan'].str.split(' ', expand=True)

# Combinar dos columnas categóricas en una sola
df_clientes['Tipo_y_Ciudad'] = df_clientes['Tipo_plan'] + '_' + df_clientes['Ciudad']

# Resumir por grupo: ingreso promedio de la ciudad de cada cliente
df_clientes['Ingreso_promedio_ciudad'] = df_clientes.groupby('Ciudad')['Ingreso'].transform('mean')

df_clientes

### 🤔 ¿Qué acaba de pasar?

- `.str.split(' ', expand=True)` corta cada texto donde encuentra un espacio y reparte los pedazos en columnas nuevas.
- `'Tipo_plan' + '_' + 'Ciudad'` simplemente pega texto, fila por fila, para capturar una combinación específica (ej. `Corporativo_Tunja`).
- `groupby('Ciudad')['Ingreso'].transform('mean')` NO reduce la tabla a una fila por ciudad (como haría `.mean()` solo): en cambio, le "copia y pega" a **cada cliente** el promedio de su propia ciudad, manteniendo el mismo número de filas.

> ⚠️ **Cuidado con la fuga de datos (*data leakage*):** si vas a dividir tus datos en entrenamiento y prueba, el promedio por grupo (como `Ingreso_promedio_ciudad`) debe calcularse **solo con los datos de entrenamiento**, y luego pegarse a los datos de prueba con un `.merge()`. Si lo calculas con la tabla completa antes de dividir, "se te cuelan" datos de prueba en el promedio, y tu evaluación del modelo queda inflada de forma artificial.

---
### ✅ Autocomprobación Rápida

**Pregunta:** Un compañero calcula el "gasto promedio por barrio" usando **todo** el dataset (entrenamiento + prueba juntos) antes de dividirlo. Luego entrena su modelo y obtiene una precisión altísima. ¿Por qué deberías desconfiar de ese resultado?

<details>
<summary>💡 Ver respuesta</summary>

Porque el promedio de cada barrio ya "vio" los valores de las filas de prueba antes de calcularse. Es decir, información del conjunto de prueba se filtró silenciosamente al de entrenamiento (*data leakage*). El modelo no está siendo evaluado de forma justa: en producción, con datos verdaderamente nuevos, su precisión real sería mucho menor. La regla correcta es calcular el promedio **solo** con el conjunto de entrenamiento y luego pegarlo (`merge`) al conjunto de prueba.

</details>

---
## 5. Resumen relámpago ⚡

| Técnica | ¿Para qué sirve? | Ejemplo |
|---|---|---|
| **Ratio** | Comparar dos columnas de forma justa | `Precio / Area` |
| **Logaritmo (`log1p`)** | Aplanar distribuciones muy sesgadas | `np.log1p(Ingreso)` |
| **Conteo de booleanos** | Resumir muchas banderas sí/no en un número | `df[cols].sum(axis=1)` |
| **Separar / combinar texto** | Extraer o fusionar información categórica | `.str.split(' ')` |
| **Resumen por grupo (`transform`)** | Darle a cada fila una estadística de su grupo | `groupby('Ciudad').transform('mean')` |

➡️ **Siguiente paso:** en el cuaderno [04 - PCA (Para Dummies)](04_PCA_Feature_Engineering_Dummies.ipynb) aprenderás a resumir muchas columnas numéricas en unas pocas "supervariables" con el Análisis de Componentes Principales (PCA).

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
